In [2]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    )

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

# ==========================================================
# 2. CONFIGURACIÓN
# ==========================================================
URL_OBJETIVO = "https://books.toscrape.com/"
SELECTOR_CONTENEDOR = "article.product_pod"
SELECTOR_TITULO = "h3 a"
SELECTOR_PRECIO = "p.price_color"

paginas_objetivo = 5
dataset_final = []

# ==========================================================
# 3. SCRAPING CON PAGINACIÓN
# ==========================================================
driver = iniciar_navegador()

try:
    print("Conectando a la fuente...")
    driver.get(URL_OBJETIVO)

    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i + 1} ---")

        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, SELECTOR_CONTENEDOR))
        )

        elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
        print(f"Se encontraron {len(elementos)} productos")

        for item in elementos:
            try:
                titulo = item.find_element(By.CSS_SELECTOR, SELECTOR_TITULO).text
                precio = item.find_element(By.CSS_SELECTOR, SELECTOR_PRECIO).text

                valor_limpio = "".join(filter(str.isdigit, precio))

                dataset_final.append({
                    "titulo": titulo,
                    "precio": precio,
                    "precio_num": valor_limpio,
                    "pagina": i + 1
                })

            except:
                continue

        # PAGINACIÓN
        try:
            boton_siguiente = driver.find_element(By.XPATH, "//li[@class='next']/a")
            boton_siguiente.click()
            time.sleep(2)

        except:
            print("Última página alcanzada.")
            break

    print("\nScraping finalizado")
    print(f"Total registros: {len(dataset_final)}")

# ==========================================================
# 4. EXPORTAR DATOS
# ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_libros.csv", index=False)

        print("Archivo generado: datos_libros.csv")
        print(df.head())
    else:
        print("No se capturaron datos")

except Exception as e:
    print(f"Error: {e}")

finally:
    driver.quit()
    print("Navegador cerrado")

Conectando a la fuente...
--- Procesando Página 1 ---
Se encontraron 20 productos
--- Procesando Página 2 ---
Se encontraron 20 productos
--- Procesando Página 3 ---
Se encontraron 20 productos
--- Procesando Página 4 ---
Se encontraron 20 productos
--- Procesando Página 5 ---
Se encontraron 20 productos

Scraping finalizado
Total registros: 100
Archivo generado: datos_libros.csv
                         titulo  precio precio_num  pagina
0            A Light in the ...  £51.77       5177       1
1            Tipping the Velvet  £53.74       5374       1
2                    Soumission  £50.10       5010       1
3                 Sharp Objects  £47.82       4782       1
4  Sapiens: A Brief History ...  £54.23       5423       1
Navegador cerrado
